In [1]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [2]:
import pandas as pd

In [11]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = os.environ.get("TMG_REPO_URL", "https://github.com/Omayer111/TMG-GAN.git")
REPO_ROOT = Path(os.environ.get("TMG_REPO_ROOT", "/content/TMG-GAN"))
BRANCH = os.environ.get("TMG_BRANCH", "main")
ALLOW_DIRTY = os.environ.get("TMG_ALLOW_DIRTY", "0") == "1"

def run(cmd, cwd=None, check=True):
    print("+", " ".join(cmd))
    p = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        text=True,
        capture_output=True,
    )
    if p.stdout:
        print(p.stdout.rstrip())
    if p.stderr:
        print(p.stderr.rstrip())
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(cmd)}")
    return p

# 1) Ensure repo exists in runtime
if not REPO_ROOT.exists():
    run(["git", "clone", REPO_URL, str(REPO_ROOT)])

# 2) Ensure origin URL is correct
run(["git", "-C", str(REPO_ROOT), "remote", "set-url", "origin", REPO_URL], check=False)

# 3) Fetch and sync
run(["git", "-C", str(REPO_ROOT), "fetch", "origin", "--prune"])

if BRANCH == "auto":
    head_ref = run(
        ["git", "-C", str(REPO_ROOT), "symbolic-ref", "refs/remotes/origin/HEAD"],
        check=False,
    ).stdout.strip()
    BRANCH = head_ref.split("/")[-1] if head_ref else "main"

status = run(["git", "-C", str(REPO_ROOT), "status", "--porcelain"], check=False).stdout.strip()
if status and not ALLOW_DIRTY:
    raise RuntimeError(
        "Runtime repo has local edits. Commit/stash/discard first, "
        "or set TMG_ALLOW_DIRTY=1 if you intentionally want to keep them."
    )

run(["git", "-C", str(REPO_ROOT), "checkout", BRANCH])
run(["git", "-C", str(REPO_ROOT), "pull", "--rebase", "origin", BRANCH])

commit = run(["git", "-C", str(REPO_ROOT), "rev-parse", "--short", "HEAD"]).stdout.strip()
print("Synced commit:", commit)

# 4) Export results root for downstream cells
os.environ["TMG_RESULTS_ROOT"] = str(REPO_ROOT / "results")
print("TMG_RESULTS_ROOT =", os.environ["TMG_RESULTS_ROOT"])

# 5) Optional sanity check: does runtime trainer support key flags?
train_script = REPO_ROOT / "results" / "thesis_tmg_pipeline" / "scripts" / "train_tmg_gan.py"
if train_script.exists():
    help_text = run([sys.executable, str(train_script), "-h"], check=False).stdout
    for flag in ["--max-fallback-rate", "--strict-qualification-fallback"]:
        print(f"{flag} supported:", flag in help_text)
else:
    print("Trainer script not found at expected location.")

+ git -C /content/TMG-GAN remote set-url origin https://github.com/Omayer111/TMG-GAN.git
fatal: not a git repository (or any of the parent directories): .git
+ git -C /content/TMG-GAN fetch origin --prune
fatal: not a git repository (or any of the parent directories): .git


RuntimeError: Command failed (128): git -C /content/TMG-GAN fetch origin --prune

In [2]:
from pathlib import Path
from google.colab import drive
import zipfile, os

drive.mount("/content/drive", force_remount=False)

# Choose one runtime root and stick to it
results_root = Path("/content/TMG-GAN/results")
results_root.mkdir(parents=True, exist_ok=True)

payload = Path("/content/drive/MyDrive/tmg_colab_payload.zip")
with zipfile.ZipFile(payload, "r") as z:
    z.extractall(results_root)

os.environ["TMG_RESULTS_ROOT"] = str(results_root)
os.environ["TMG_ALLOW_DRIVE_MOUNT"] = "0"  # keep your no-write behavior
print("TMG_RESULTS_ROOT =", os.environ["TMG_RESULTS_ROOT"])

Mounted at /content/drive
TMG_RESULTS_ROOT = /content/TMG-GAN/results


In [3]:
from pathlib import Path
r = Path("/content/TMG-GAN/results")
checks = {
    "train script": r / "thesis_tmg_pipeline/scripts/train_tmg_gan.py",
    "dataset dir": r / "datasets/CICIDS2017",
    "checkpoint": r / "outputs_tmg/checkpoints/CICIDS2017/tmg_gan_tabular/latest.pt",
}
for k, p in checks.items():
    print(f"{k}: {p.exists()} -> {p}")

train script: True -> /content/TMG-GAN/results/thesis_tmg_pipeline/scripts/train_tmg_gan.py
dataset dir: True -> /content/TMG-GAN/results/datasets/CICIDS2017
checkpoint: True -> /content/TMG-GAN/results/outputs_tmg/checkpoints/CICIDS2017/tmg_gan_tabular/latest.pt


In [4]:
import os, subprocess
from pathlib import Path

os.environ["TMG_ALLOW_DRIVE_MOUNT"] = "0"
os.environ["TMG_REPO_URL"] = "https://github.com/Omayer111/TMG-GAN.git"
os.environ["TMG_CLONE_BASE"] = "/content"

repo_root = Path("/content/TMG-GAN")
if not repo_root.exists():
    subprocess.run(["git", "clone", os.environ["TMG_REPO_URL"], str(repo_root)], check=True)

os.environ["TMG_RESULTS_ROOT"] = "/content/TMG-GAN/results"
print("TMG_RESULTS_ROOT =", os.environ["TMG_RESULTS_ROOT"])

TMG_RESULTS_ROOT = /content/TMG-GAN/results


In [5]:
from pathlib import Path
r = Path("/content/TMG-GAN/results")
checks = {
    "train script": r / "thesis_tmg_pipeline/scripts/train_tmg_gan.py",
    "dataset dir": r / "datasets/CICIDS2017",
    "checkpoint": r / "outputs_tmg/checkpoints/CICIDS2017/tmg_gan_tabular/latest.pt",
}
for k, p in checks.items():
    print(f"{k}: {p.exists()} -> {p}")

train script: True -> /content/TMG-GAN/results/thesis_tmg_pipeline/scripts/train_tmg_gan.py
dataset dir: True -> /content/TMG-GAN/results/datasets/CICIDS2017
checkpoint: True -> /content/TMG-GAN/results/outputs_tmg/checkpoints/CICIDS2017/tmg_gan_tabular/latest.pt


In [6]:
# Step 1/6: Colab runtime setup and path resolution
from pathlib import Path, PureWindowsPath
from datetime import datetime
import json
import os
import subprocess
import sys

import torch

PY = sys.executable
DATASET = "CICIDS2017"
RUN_STEM = "tmg_gan_tabular"
RUN_NAME_OVERRIDE = os.environ.get("TMG_RUN_NAME", "").strip()
RUN_TAG = os.environ.get("TMG_RUN_TAG", "").strip()
RESUME_MODE = os.environ.get("TMG_RESUME", "0") == "1"
BASELINE_RUN_NAME = os.environ.get("TMG_BASELINE_RUN_NAME", RUN_STEM).strip() or RUN_STEM
REQUIRED_DATA_FILES = ("x_train.csv", "y_train.csv", "x_test.csv", "y_test.csv")

if RUN_NAME_OVERRIDE:
    RUN_NAME = RUN_NAME_OVERRIDE
elif RUN_TAG:
    RUN_NAME = f"{RUN_STEM}_{RUN_TAG}"
else:
    RUN_NAME = f"{RUN_STEM}_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}"

# Explicit Windows path (used only when runtime can access it).
WIN_RESULTS_ROOT = r"C:\Users\USER\Desktop\TMG-GAN\TMG-GAN\results"

# Runtime overrides (recommended for Colab from VS Code).
RUNTIME_RESULTS_ROOT = os.environ.get("TMG_RESULTS_ROOT", "").strip() or None
TMG_REPO_URL = os.environ.get("TMG_REPO_URL", "https://github.com/Omayer111/TMG-GAN.git")
CLONE_BASE = Path(os.environ.get("TMG_CLONE_BASE", "/content"))
ALLOW_DRIVE_MOUNT = os.environ.get("TMG_ALLOW_DRIVE_MOUNT", "0") == "1"

def _maybe_mount_google_drive() -> bool:
    try:
        from google.colab import drive  # type: ignore
    except Exception:
        return False

    my_drive = Path("/content/drive/MyDrive")
    if my_drive.exists():
        return True

    try:
        drive.mount("/content/drive", force_remount=False)
    except Exception as exc:
        print(f"Drive mount skipped: {exc}")
        return False
    return my_drive.exists()

def _windows_path_candidates(raw_path: str):
    candidates = [Path(raw_path), Path(raw_path.replace("\\", "/"))]
    win = PureWindowsPath(raw_path)
    if win.drive:
        drive = win.drive.replace(":", "").lower()
        tail = Path(*win.parts[1:]) if len(win.parts) > 1 else Path()
        candidates.append(Path("/mnt") / drive / tail)
    return candidates

def _is_valid_results_root(results_root: Path) -> bool:
    return (
        (results_root / "thesis_tmg_pipeline" / "scripts" / "train_tmg_gan.py").exists()
        and (results_root / "datasets" / DATASET).exists()
    )

def _candidate_results_roots():
    raw_inputs = [WIN_RESULTS_ROOT]
    if RUNTIME_RESULTS_ROOT:
        raw_inputs.insert(0, RUNTIME_RESULTS_ROOT)

    for raw in raw_inputs:
        for candidate in _windows_path_candidates(raw):
            yield candidate

    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        yield base
        yield base / "results"

    common = [
        Path("/content/TMG-GAN/results"),
        Path("/content/TMG-GAN/TMG-GAN/results"),
        Path("/content/drive/MyDrive/TMG-GAN/results"),
        Path("/content/drive/MyDrive/TMG-GAN/TMG-GAN/results"),
        Path("/content/drive/MyDrive/results"),
        Path("/kaggle/working/TMG-GAN/results"),
        Path("/kaggle/working/TMG-GAN/TMG-GAN/results"),
        Path("/kaggle/working/results"),
        Path("/content/results"),
    ]
    for candidate in common:
        yield candidate

def _search_results_root():
    checked = []
    seen = set()

    for candidate in _candidate_results_roots():
        key = str(candidate)
        if key in seen:
            continue
        seen.add(key)
        checked.append(key)
        if _is_valid_results_root(candidate):
            return candidate.resolve(), checked

    search_roots = [Path.cwd().resolve(), Path("/content"), Path("/kaggle/working"), Path.home()]
    patterns = [
        "**/results/thesis_tmg_pipeline/scripts/train_tmg_gan.py",
        "**/thesis_tmg_pipeline/scripts/train_tmg_gan.py",
    ]

    for root in search_roots:
        if not root.exists():
            continue
        for pattern in patterns:
            for hit in root.glob(pattern):
                if hit.parent.parent.parent.name == "results":
                    candidate = hit.parent.parent.parent
                else:
                    candidate = hit.parent.parent.parent.parent
                checked.append(str(candidate))
                if _is_valid_results_root(candidate):
                    return candidate.resolve(), checked

    return None, checked

def _bootstrap_repo_into_runtime():
    clone_root = CLONE_BASE / "TMG-GAN"
    if not clone_root.exists():
        clone_root.parent.mkdir(parents=True, exist_ok=True)
        print(f"Cloning repo for runtime: {TMG_REPO_URL} -> {clone_root}")
        subprocess.run(["git", "clone", TMG_REPO_URL, str(clone_root)], check=True)
    elif (clone_root / ".git").exists():
        print(f"Refreshing repo in runtime: {clone_root}")
        subprocess.run(["git", "-C", str(clone_root), "pull", "--ff-only"], check=False)

    for candidate in (clone_root / "results", clone_root / "TMG-GAN" / "results"):
        if _is_valid_results_root(candidate):
            return candidate.resolve()

    return None

def _find_results_root():
    resolved, checked = _search_results_root()
    if resolved is not None:
        return resolved, checked

    if ALLOW_DRIVE_MOUNT and _maybe_mount_google_drive():
        resolved, checked_after_mount = _search_results_root()
        checked.extend(checked_after_mount)
        if resolved is not None:
            return resolved, checked

    bootstrapped = _bootstrap_repo_into_runtime()
    if bootstrapped is not None:
        checked.append(str(bootstrapped))
        return bootstrapped, checked

    raise FileNotFoundError(
        "Could not locate a valid results root in this runtime. "
        "Expected layout: <results>/thesis_tmg_pipeline/scripts/train_tmg_gan.py and "
        f"<results>/datasets/{DATASET}. "
        "Drive may be mounted but repo is missing there. "
        f"Tried repo bootstrap URL: {TMG_REPO_URL}. "
        "You can override with env TMG_RESULTS_ROOT. "
        "Checked: " + " | ".join(checked)
    )

RESULTS_ROOT, results_root_checked = _find_results_root()
PROJECT_ROOT = RESULTS_ROOT / "thesis_tmg_pipeline"
DATASET_DIR = RESULTS_ROOT / "datasets" / DATASET
CHECKPOINT_DIR = RESULTS_ROOT / "outputs_tmg" / "checkpoints" / DATASET / RUN_NAME
OUT_DIR = RESULTS_ROOT / "outputs_tmg"
CACHE_DIR = RESULTS_ROOT / "data_cache"

project_root = PROJECT_ROOT
results_root = RESULTS_ROOT
DATA_ROOT = DATASET_DIR.parent

requirements_file = PROJECT_ROOT / "requirements.txt"
preflight_script = PROJECT_ROOT / "scripts" / "colab_preflight.py"
train_script = PROJECT_ROOT / "scripts" / "train_tmg_gan.py"
visualize_script = PROJECT_ROOT / "scripts" / "visualize_run_metrics.py"
metrics_path = OUT_DIR / "metrics" / DATASET / f"{RUN_NAME}.json"
baseline_metrics_path = OUT_DIR / "metrics" / DATASET / f"{BASELINE_RUN_NAME}.json"
ckpt_latest = CHECKPOINT_DIR / "latest.pt"
log_dir = OUT_DIR / "logs"
log_mode = "resume" if RESUME_MODE else "fresh"
log_file = log_dir / f"{RUN_NAME}_{log_mode}.log"

os.chdir(PROJECT_ROOT)

print(f"Runtime cwd: {Path.cwd().resolve()}")
print(f"Results-root candidates checked: {len(results_root_checked)}")
print(f"Results root: {results_root}")
print(f"Project root: {project_root}")
print(f"Dataset dir: {DATASET_DIR}")
print(f"Output dir: {OUT_DIR}")
print(f"Cache dir: {CACHE_DIR}")
print(f"Run mode: {'RESUME' if RESUME_MODE else 'FRESH'}")
print(f"Run name: {RUN_NAME}")
print(f"Baseline run for comparison: {BASELINE_RUN_NAME}")
print(f"Checkpoint dir: {CHECKPOINT_DIR}")

Runtime cwd: /content/TMG-GAN/results/thesis_tmg_pipeline
Results-root candidates checked: 1
Results root: /content/TMG-GAN/results
Project root: /content/TMG-GAN/results/thesis_tmg_pipeline
Dataset dir: /content/TMG-GAN/results/datasets/CICIDS2017
Output dir: /content/TMG-GAN/results/outputs_tmg
Cache dir: /content/TMG-GAN/results/data_cache
Run mode: FRESH
Run name: tmg_gan_tabular_20260310_075419
Baseline run for comparison: tmg_gan_tabular
Checkpoint dir: /content/TMG-GAN/results/outputs_tmg/checkpoints/CICIDS2017/tmg_gan_tabular_20260310_075419


/tmp/ipykernel_645/2027399658.py:25: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  RUN_NAME = f"{RUN_STEM}_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}"


In [7]:
# Step 2/6: Validate paths, dataset files, and checkpoint (if resuming)
for required_path in (RESULTS_ROOT, PROJECT_ROOT, DATASET_DIR):
    if not required_path.exists():
        raise FileNotFoundError(f"Required path not found: {required_path}")

for required_file in (requirements_file, preflight_script, train_script, visualize_script):
    if not required_file.exists():
        raise FileNotFoundError(f"Required file not found: {required_file}")

missing = [name for name in REQUIRED_DATA_FILES if not (DATASET_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Missing dataset files in {DATASET_DIR}: {', '.join(missing)}")

OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)

if RESUME_MODE and (not ckpt_latest.exists()):
    raise FileNotFoundError(
        f"Resume checkpoint not found: {ckpt_latest}. "
        "Set TMG_RESUME=0 for a fresh run, or provide TMG_RUN_NAME pointing to a run with latest.pt."
    )

print("Validation OK")
print(f"Checkpoint expected: {ckpt_latest}")
print(f"Log file: {log_file}")

Validation OK
Checkpoint expected: /content/TMG-GAN/results/outputs_tmg/checkpoints/CICIDS2017/tmg_gan_tabular_20260310_075419/latest.pt
Log file: /content/TMG-GAN/results/outputs_tmg/logs/tmg_gan_tabular_20260310_075419_fresh.log


In [8]:
# Step 3/6: Define helpers and read baseline metric
def read_best_f1(path: Path):
    if not path.exists():
        return None
    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
        return float(payload.get("best_f1"))
    except Exception:
        return None

def stream_command_with_log(cmd, log_path: Path, cwd: Path = None) -> None:
    print("Running:")
    print(" ".join(cmd))
    with open(log_path, "a", encoding="utf-8", buffering=1) as log_fp:
        log_fp.write("\n" + "=" * 80 + "\n")
        log_fp.write(f"[{datetime.utcnow().isoformat()}Z] START\n")
        log_fp.write(" ".join(cmd) + "\n")
        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            cwd=str(cwd) if cwd is not None else None,
        )
        for line in process.stdout:
            print(line, end="")
            log_fp.write(line)
        process.wait()
        log_fp.write(f"[{datetime.utcnow().isoformat()}Z] END rc={process.returncode}\n")
    if process.returncode != 0:
        raise subprocess.CalledProcessError(process.returncode, cmd)

def print_log_tail(path: Path, n: int = 40):
    if not path.exists():
        return
    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    print("\n----- Log Tail -----")
    for line in lines[-n:]:
        print(line)
    print("----- End Log Tail -----")

comparison_metrics_path = metrics_path if RESUME_MODE else baseline_metrics_path
baseline_best_f1 = read_best_f1(comparison_metrics_path)
print(f"Comparison metrics file: {comparison_metrics_path}")
print(f"Baseline best_f1: {baseline_best_f1}")

Comparison metrics file: /content/TMG-GAN/results/outputs_tmg/metrics/CICIDS2017/tmg_gan_tabular.json
Baseline best_f1: None


In [9]:
# Step 4/6: Install dependencies and run preflight
subprocess.run([PY, "-m", "pip", "install", "-r", str(requirements_file)], check=True, cwd=str(PROJECT_ROOT))

preflight_cmd = [
    PY,
    str(preflight_script),
    "--dataset",
    DATASET,
    "--data-root",
    str(DATA_ROOT),
    "--output-dir",
    str(OUT_DIR),
    "--cache-dir",
    str(CACHE_DIR),
    "--run-name",
    RUN_NAME,
]
if RESUME_MODE:
    preflight_cmd.append("--resume")
subprocess.run(preflight_cmd, check=True, cwd=str(PROJECT_ROOT))
print("Preflight OK")

Preflight OK


In [10]:
# Step 5/6: Train (fresh or resume) with robust defaults
AUG_TARGET_MODE = os.environ.get("TMG_AUG_TARGET_MODE", "second_max").strip() or "second_max"
MAX_SYN_MULTIPLIER = float(os.environ.get("TMG_MAX_SYN_MULT", "1.5"))
CLF_WEIGHT_MODE = os.environ.get("TMG_CLF_WEIGHTING", "none").strip() or "none"
STRICT_QUAL = os.environ.get("TMG_STRICT_QUAL", "1") == "1"
MAX_FALLBACK_RATE = float(os.environ.get("TMG_MAX_FALLBACK_RATE", "0.05"))

if RESUME_MODE:
    ckpt = torch.load(ckpt_latest, map_location="cpu", weights_only=False)
    phase = str(ckpt.get("phase", "gan"))
    gan_epoch = int(ckpt.get("gan_epoch", -1)) + 1
    clf_epoch = int(ckpt.get("clf_epoch", -1)) + 1

    gan_cfg = ckpt.get("gan_config", {})
    cfg = ckpt.get("config", {})
    ckpt_best_f1 = float(ckpt.get("best_f1", float("nan")))

    # Recover exact architecture used by checkpoint.
    resume_hidden_dim = int(
        gan_cfg.get(
            "gan_hidden_dim",
            cfg.get(
                "hidden_dim",
                ckpt["cd_model_state_dict"]["backbone.0.parametrizations.weight.original"].shape[0],
            ),
        )
    )
    resume_z_dim = int(
        gan_cfg.get(
            "z_dim",
            ckpt["generator_state_dicts"][0]["backbone.0.weight"].shape[1],
        )
    )

    target_gan_epochs = max(gan_epoch + 100, gan_epoch + 1) if phase == "gan" else max(1, gan_epoch)
    target_clf_epochs = max(clf_epoch + 100, clf_epoch + 1) if phase == "clf" else 200

    resume_args = [
        "--resume",
        "--robust-rng-restore",
        "--reset-clf-optimizer-on-resume",
    ]

    print("Resume metadata:", {
        "phase": phase,
        "gan_epoch_start": gan_epoch,
        "clf_epoch_start": clf_epoch,
        "best_f1": ckpt_best_f1,
    })
    print("Resume architecture:", {"gan_hidden_dim": resume_hidden_dim, "z_dim": resume_z_dim})
    print("Resume targets:", {"gan_epochs": target_gan_epochs, "clf_epochs": target_clf_epochs})
else:
    phase = "gan"
    resume_hidden_dim = int(os.environ.get("TMG_GAN_HIDDEN_DIM", "256"))
    resume_z_dim = int(os.environ.get("TMG_Z_DIM", "64"))
    target_gan_epochs = int(os.environ.get("TMG_GAN_EPOCHS", "300"))
    target_clf_epochs = int(os.environ.get("TMG_CLF_EPOCHS", "200"))
    resume_args = []

    print("Fresh run configuration:", {
        "gan_hidden_dim": resume_hidden_dim,
        "z_dim": resume_z_dim,
        "gan_epochs": target_gan_epochs,
        "clf_epochs": target_clf_epochs,
    })

train_cmd = [
    PY,
    "-u",
    str(train_script),
    "--dataset",
    DATASET,
    "--data-root",
    str(DATA_ROOT),
    "--output-dir",
    str(OUT_DIR),
    "--cache-dir",
    str(CACHE_DIR),
    "--run-name",
    RUN_NAME,
    "--gan-hidden-dim",
    str(resume_hidden_dim),
    "--z-dim",
    str(resume_z_dim),
    "--gan-epochs",
    str(target_gan_epochs),
    "--epochs",
    str(target_clf_epochs),
    "--gan-lr",
    "2e-4",
    "--cd-steps",
    "1",
    "--g-steps",
    "1",
    "--batch-size",
    "1024",
    "--gen-batch-size",
    "2048",
    "--hidden-warmup-epochs",
    "100",
    "--hidden-loss-weight",
    "1.0",
    "--gan-eval-interval",
    "10",
    "--augmentation-cap",
    "300000",
    "--augmentation-target-mode",
    AUG_TARGET_MODE,
    "--max-synthetic-multiplier",
    str(MAX_SYN_MULTIPLIER),
    "--max-rejects",
    "20",
    "--max-fallback-rate",
    str(MAX_FALLBACK_RATE),
    "--clf-class-weighting",
    CLF_WEIGHT_MODE,
    "--clf-effective-num-beta",
    "0.9999",
    "--clf-label-smoothing",
    "0.02",
    "--clf-lr-patience",
    "5",
    "--clf-lr-decay",
    "0.5",
    "--clf-min-lr",
    "3e-5",
    "--clf-early-stop-patience",
    "20",
    "--checkpoint-interval",
    "1",
    "--eval-interval",
    "1",
    *resume_args,
]

if STRICT_QUAL:
    train_cmd.append("--strict-qualification-fallback")

stream_command_with_log(train_cmd, log_file, cwd=PROJECT_ROOT)

Fresh run configuration: {'gan_hidden_dim': 256, 'z_dim': 64, 'gan_epochs': 300, 'clf_epochs': 200}
Running:
/usr/bin/python3 -u /content/TMG-GAN/results/thesis_tmg_pipeline/scripts/train_tmg_gan.py --dataset CICIDS2017 --data-root /content/TMG-GAN/results/datasets --output-dir /content/TMG-GAN/results/outputs_tmg --cache-dir /content/TMG-GAN/results/data_cache --run-name tmg_gan_tabular_20260310_075419 --gan-hidden-dim 256 --z-dim 64 --gan-epochs 300 --epochs 200 --gan-lr 2e-4 --cd-steps 1 --g-steps 1 --batch-size 1024 --gen-batch-size 2048 --hidden-warmup-epochs 100 --hidden-loss-weight 1.0 --gan-eval-interval 10 --augmentation-cap 300000 --augmentation-target-mode second_max --max-synthetic-multiplier 1.5 --max-rejects 20 --max-fallback-rate 0.05 --clf-class-weighting none --clf-effective-num-beta 0.9999 --clf-label-smoothing 0.02 --clf-lr-patience 5 --clf-lr-decay 0.5 --clf-min-lr 3e-5 --clf-early-stop-patience 20 --checkpoint-interval 1 --eval-interval 1 --strict-qualification-fal

/tmp/ipykernel_645/4084730645.py:16: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  log_fp.write(f"[{datetime.utcnow().isoformat()}Z] START\n")


usage: train_tmg_gan.py [-h] --dataset {CICIDS2017,UNSW-NB15} --data-root
                        DATA_ROOT [--cache-dir CACHE_DIR]
                        [--output-dir OUTPUT_DIR] [--run-name RUN_NAME]
                        [--epochs EPOCHS] [--batch-size BATCH_SIZE] [--lr LR]
                        [--seed SEED] [--gan-epochs GAN_EPOCHS]
                        [--gan-lr GAN_LR] [--z-dim Z_DIM]
                        [--gan-hidden-dim GAN_HIDDEN_DIM]
                        [--cd-steps CD_STEPS] [--g-steps G_STEPS]
                        [--gen-batch-size GEN_BATCH_SIZE]
                        [--hidden-warmup-epochs HIDDEN_WARMUP_EPOCHS]
                        [--hidden-loss-weight HIDDEN_LOSS_WEIGHT]
                        [--max-rejects MAX_REJECTS]
                        [--gan-eval-interval GAN_EVAL_INTERVAL]
                        [--max-grad-norm MAX_GRAD_NORM]
                        [--augmentation-cap AUGMENTATION_CAP]
                        [--augmentation-targ

/tmp/ipykernel_645/4084730645.py:30: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  log_fp.write(f"[{datetime.utcnow().isoformat()}Z] END rc={process.returncode}\n")


CalledProcessError: Command '['/usr/bin/python3', '-u', '/content/TMG-GAN/results/thesis_tmg_pipeline/scripts/train_tmg_gan.py', '--dataset', 'CICIDS2017', '--data-root', '/content/TMG-GAN/results/datasets', '--output-dir', '/content/TMG-GAN/results/outputs_tmg', '--cache-dir', '/content/TMG-GAN/results/data_cache', '--run-name', 'tmg_gan_tabular_20260310_075419', '--gan-hidden-dim', '256', '--z-dim', '64', '--gan-epochs', '300', '--epochs', '200', '--gan-lr', '2e-4', '--cd-steps', '1', '--g-steps', '1', '--batch-size', '1024', '--gen-batch-size', '2048', '--hidden-warmup-epochs', '100', '--hidden-loss-weight', '1.0', '--gan-eval-interval', '10', '--augmentation-cap', '300000', '--augmentation-target-mode', 'second_max', '--max-synthetic-multiplier', '1.5', '--max-rejects', '20', '--max-fallback-rate', '0.05', '--clf-class-weighting', 'none', '--clf-effective-num-beta', '0.9999', '--clf-label-smoothing', '0.02', '--clf-lr-patience', '5', '--clf-lr-decay', '0.5', '--clf-min-lr', '3e-5',

In [ ]:
# Step 6/6: Render metrics and print verdict
subprocess.run(
    [
        PY,
        str(visualize_script),
        "--dataset",
        DATASET,
        "--output-dir",
        str(OUT_DIR),
        "--run-name",
        RUN_NAME,
    ],
    check=True,
    cwd=str(PROJECT_ROOT),
)

new_best_f1 = read_best_f1(metrics_path)
print(f"New best_f1: {new_best_f1}")
if baseline_best_f1 is None:
    print("Verdict: baseline missing; run completed and produced fresh metrics.")
elif new_best_f1 is None:
    print("Verdict: run completed but could not parse new best_f1.")
else:
    delta = new_best_f1 - baseline_best_f1
    if delta > 0:
        print(f"Verdict: IMPROVED (delta={delta:.6f}).")
    elif delta < 0:
        print(f"Verdict: NOT IMPROVED (delta={delta:.6f}).")
    else:
        print("Verdict: UNCHANGED (delta=0).")

print_log_tail(log_file, n=40)
print(f"Full log file for monitoring: {log_file}")